# 02 Fault Robustness

This notebook explores controller behavior under actuator degradation and rotor-loss scenarios.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from src.configuration import load_json_config
from src.control import build_controller
from src.dynamics import VTOLDynamicsModel
from src.faults import FaultInjector
from src.plotting import plot_comparison, plot_simulation_results
from src.simulations.core import simulate_experiment
from src.simulations.trajectories import hover_trajectory


In [ ]:
config = load_json_config(REPO_ROOT / "config" / "default_params.json")
model = VTOLDynamicsModel.from_config(config)
dt = config["simulations"]["dt"]
duration = config["simulations"]["duration_s"]
initial_state = np.zeros(12)
initial_state[2] = -2.0

fault = FaultInjector.from_dict(config["fault_examples"]["single_rotor_hover"], model.rotor_count)
pid_results = simulate_experiment(model, build_controller("pid", model, config), fault, hover_trajectory(position=(0.0, 0.0, -2.0)), initial_state, duration, dt)
lqr_results = simulate_experiment(model, build_controller("lqr", model, config), fault, hover_trajectory(position=(0.0, 0.0, -2.0)), initial_state, duration, dt)
plot_comparison(pid_results, lqr_results, title="Single rotor failure: PID vs LQR")


In [ ]:
plot_simulation_results(pid_results, title="PID under single-rotor failure")
plot_simulation_results(lqr_results, title="LQR under single-rotor failure")


**Research note:** The present scaffold is useful for ranking baseline resilience trends, but true fault-tolerant claims will require controller redesign plus a more faithful transition model, mixer constraints, and estimator dynamics.
